# 02 — Annotation EDA

Run after `python -m src.annotate_cli` (and optionally `python -m src.annotate_llm`) to inspect the labels and check inter-annotator agreement.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))
import pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import cohen_kappa_score, confusion_matrix
from src.paths import ANN_HUMAN, ANN_LLM, LABELS

In [ ]:
h = pd.read_csv(ANN_HUMAN)
print(f'Human annotations: {len(h)}')
h['label'].value_counts().reindex(LABELS+['skip']).plot(kind='bar', color='#7eb0ff')
plt.title('Human label distribution'); plt.xticks(rotation=0); plt.show()
print('\nMedian time/sample (s):', h['annot_time_sec'].median())

In [ ]:
if ANN_LLM.exists():
    llm = pd.read_csv(ANN_LLM)
    merged = h[h['label'].isin(LABELS)][['id','label']].merge(
        llm[llm['label'].isin(LABELS)][['id','label']],
        on='id', suffixes=('_human','_llm'))
    print(f'Overlap: {len(merged)}')
    print(f"Cohen's κ = {cohen_kappa_score(merged['label_human'], merged['label_llm'], labels=LABELS):.3f}")
    print(f"raw agreement = {(merged['label_human']==merged['label_llm']).mean():.3f}")
    cm = confusion_matrix(merged['label_human'], merged['label_llm'], labels=LABELS)
    plt.imshow(cm, cmap='Blues'); plt.xticks(range(3), LABELS); plt.yticks(range(3), LABELS)
    plt.xlabel('LLM'); plt.ylabel('human'); plt.title('Human vs LLM confusion')
    for i in range(3):
        for j in range(3):
            plt.text(j, i, int(cm[i,j]), ha='center', va='center')
    plt.show()
else:
    print('No LLM annotations yet. Run python -m src.annotate_llm on Helios.')

In [ ]:
# Look at examples where human says hate but the LLM says non_hate (and vice versa)
if ANN_LLM.exists():
    merged2 = h.merge(llm[['id','label']].rename(columns={'label':'llm_label'}), on='id')
    disagree = merged2[(merged2['label'] != merged2['llm_label']) & merged2['label'].isin(LABELS)]
    for _, r in disagree.sample(min(8, len(disagree)), random_state=0).iterrows():
        print(f'human={r["label"]:10} llm={r["llm_label"]:10} | {r["text"][:150]}')
        print()